# POLARIS-X — Polarimetric Stress Imaging (Simulation V1.1)

**For a supervisor demonstration.** This runnable notebook produces mechanical stress-difference maps, phase-retardation maps, idealized polarization-camera images, and a rotating-analyzer comparison.

## Precisely what this simulation is

- **Mechanical model:** classical **Kirsch analytical solution** for an *infinite* elastic plate with a circular hole under uniform far-field tension. We plot only a 100 mm × 40 mm window. This is **not a finite-width plate solution and not ANSYS/FEA**.
- **Optical model:** the stress-optic law followed by an **ideal Jones-matrix linear-retarder model** and an ideal rotating linear analyzer.
- **Illustrative parameters:** 5 mm transparent specimen thickness and assumed stress-optic coefficient `C = 3 × 10⁻¹² Pa⁻¹`. This coefficient is **not calibrated** for a particular PMMA sample.
- **Images:** ideal monochromatic 630/525/450 nm channels combined into a synthetic RGB image. This is **not an experimentally calibrated RGB camera**.
- **Hole:** treated as clear space (no birefringence). Therefore under crossed polarizers it is dark, and under parallel polarizers it is bright. A real experiment may also show mechanical mounts, scattering, shadows, and camera artifacts.

For an easy-to-explain nominal loading reference, we calculate the far-field stress from `force_N / (40 mm × 5 mm)` (e.g., **1000 N ↔ 5 MPa**). This is only a convenient conversion; the infinite-plate Kirsch solution does **not** simulate a literal 1000 N gripping a finite 100 × 40 mm plate.

### Governing equations

\[\Delta\sigma = \sigma_1-\sigma_2 = \sqrt{(\sigma_x-\sigma_y)^2+4\tau_{xy}^2}\]

\[\delta(x,y;\lambda) = \frac{2\pi d C}{\lambda}\,\Delta\sigma(x,y)\]

For input light polarized along *x* and an output analyzer rotated by angle \(\alpha\), we evaluate the Jones vector of a local linearly birefringent element and then project it onto the analyzer. In the **crossed** case (\(\alpha=90^\circ\)) this simplifies to:

\[I/I_0 = \sin^2(2\phi)\sin^2(\delta/2)\]

where \(\phi\) is the principal-axis orientation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from pathlib import Path

plt.rcParams.update({'font.size': 10, 'figure.dpi': 115})
print('NumPy:', np.__version__)
print('Output folder:', Path.cwd())

## Set parameters and generate stress, retardation and RGB intensity maps

In [ ]:
# --- USER PARAMETERS: change these, then rerun all cells ---
WINDOW_LENGTH_MM = 100.0  # DISPLAY window only, not a finite-body FEA geometry
WINDOW_WIDTH_MM = 40.0
HOLE_RADIUS_MM = 5.0
THICKNESS_MM = 5.0
REFERENCE_WIDTH_MM = 40.0
FORCE_N = 1000.0         # nominal reference: 1000 N -> 5 MPa remote stress
ANALYZER_DEG = 90.0      # 90=crossed, 0=parallel
STRESS_OPTIC_COEFFICIENT_PA_INV = 3.0e-12  # assumed, NOT calibrated
WAVELENGTHS_NM = (630.0, 525.0, 450.0)

# Display-only brightness adjustment; no change to the physical intensity values.
DISPLAY_GAMMA = 0.65    # 1.0 means no adjustment; <1 brightens dark fringes


def kirsch_stress_field(force_N=FORCE_N, nx=551, ny=241):
    """Return ideal infinite-plate stress field, coordinates in mm, stresses in MPa."""
    if not np.isfinite(force_N) or force_N < 0:
        raise ValueError('force_N must be a finite, nonnegative number')
    x = np.linspace(-WINDOW_LENGTH_MM/2, WINDOW_LENGTH_MM/2, nx)
    y = np.linspace(-WINDOW_WIDTH_MM/2, WINDOW_WIDTH_MM/2, ny)
    X, Y = np.meshgrid(x, y)
    r = np.hypot(X, Y)
    theta = np.arctan2(Y, X)
    a = HOLE_RADIUS_MM
    outside = r >= a
    r_safe = np.maximum(r, a)
    q = (a/r_safe)**2
    remote_mpa = force_N / (REFERENCE_WIDTH_MM * THICKNESS_MM)

    sigma_rr = (remote_mpa/2)*(1-q) + (remote_mpa/2)*(1-4*q+3*q*q)*np.cos(2*theta)
    sigma_tt = (remote_mpa/2)*(1+q) - (remote_mpa/2)*(1+3*q*q)*np.cos(2*theta)
    tau_rt = -(remote_mpa/2)*(1+2*q-3*q*q)*np.sin(2*theta)

    ct, st = np.cos(theta), np.sin(theta)
    sigma_x = sigma_rr*ct**2 + sigma_tt*st**2 - 2*tau_rt*st*ct
    sigma_y = sigma_rr*st**2 + sigma_tt*ct**2 + 2*tau_rt*st*ct
    tau_xy = (sigma_rr-sigma_tt)*st*ct + tau_rt*(ct**2-st**2)
    delta_sigma = np.sqrt((sigma_x-sigma_y)**2+4*tau_xy**2)
    phi = 0.5*np.arctan2(2*tau_xy, sigma_x-sigma_y)

    # Hole contains air (idealized clear opening), so it does not produce stress birefringence.
    for field in (sigma_x, sigma_y, tau_xy, delta_sigma, phi):
        field[~outside] = 0.0
    return dict(x=x,y=y,X=X,Y=Y,outside=outside,remote_mpa=remote_mpa,
                sigma_x=sigma_x,sigma_y=sigma_y,tau_xy=tau_xy,
                delta_sigma=delta_sigma,phi=phi)


def optical_images(field, analyzer_deg=ANALYZER_DEG):
    """Ideal Jones-model RGB output; returns raw *linear* intensity on [0,1]."""
    if not np.isfinite(analyzer_deg):
        raise ValueError('analyzer_deg must be finite')
    alpha = np.deg2rad(analyzer_deg)
    phi = field['phi']
    sigma_pa = field['delta_sigma'] * 1e6
    c, s = np.cos(phi), np.sin(phi)
    images = []
    retardances = []
    for lam_nm in WAVELENGTHS_NM:
        D = 2*np.pi * (THICKNESS_MM*1e-3) * STRESS_OPTIC_COEFFICIENT_PA_INV * sigma_pa / (lam_nm*1e-9)
        # Local birefringent element: J=R(phi) diag(e^{+i D/2},e^{-i D/2}) R(-phi)
        e_plus = np.exp(0.5j*D)
        e_minus = np.exp(-0.5j*D)
        Ex = c*c*e_plus + s*s*e_minus
        Ey = c*s*(e_plus-e_minus)
        projected = np.cos(alpha)*Ex + np.sin(alpha)*Ey
        intensity = np.abs(projected)**2
        images.append(np.clip(intensity.real,0.0,1.0))
        retardances.append(D)
    return dict(rgb_linear=np.stack(images,axis=2),retardances=retardances)


def add_hole_outline(ax):
    ax.add_patch(Circle((0,0),HOLE_RADIUS_MM,fill=False,ls='--',lw=1.3,ec='white',alpha=0.8))


def produce_main_figure(force_N=FORCE_N, analyzer_deg=ANALYZER_DEG, filename='POLARIS_X_Main_Results.png'):
    field = kirsch_stress_field(force_N)
    output = optical_images(field,analyzer_deg)
    rgb = output['rgb_linear']
    extent=(-WINDOW_LENGTH_MM/2,WINDOW_LENGTH_MM/2,-WINDOW_WIDTH_MM/2,WINDOW_WIDTH_MM/2)
    fig,axes=plt.subplots(2,3,figsize=(15,8),layout='constrained')
    a=axes.ravel()
    stress=np.ma.masked_where(~field['outside'],field['delta_sigma'])
    im=a[0].imshow(stress,extent=extent,origin='lower',cmap='inferno')
    fig.colorbar(im,ax=a[0],label='MPa',shrink=.78)
    a[0].set_title('Principal stress difference (analytical)')
    dgreen=np.ma.masked_where(~field['outside'],output['retardances'][1])
    im=a[1].imshow(dgreen,extent=extent,origin='lower',cmap='viridis')
    fig.colorbar(im,ax=a[1],label='radians',shrink=.78)
    a[1].set_title('Phase retardation (525 nm)')
    a[2].imshow(rgb**DISPLAY_GAMMA,extent=extent,origin='lower',vmin=0,vmax=1)
    a[2].set_title(f'Synthetic RGB (analyzer {analyzer_deg:g}°, display γ={DISPLAY_GAMMA})')
    for idx,lam in enumerate(WAVELENGTHS_NM):
        a[idx+3].imshow(rgb[:,:,idx]**DISPLAY_GAMMA,extent=extent,origin='lower',cmap='gray',vmin=0,vmax=1)
        a[idx+3].set_title(f'{lam:g} nm: normalized intensity (display γ={DISPLAY_GAMMA})')
    for ax in a:
        ax.set_xlabel('x (mm)');ax.set_ylabel('y (mm)')
        add_hole_outline(ax)
    fig.suptitle(f'POLARIS-X V1.1 | nominal {force_N:g} N ≡ {field["remote_mpa"]:g} MPa | NOT FEA / NOT experimental',fontsize=13)
    fig.savefig(filename,dpi=155,bbox_inches='tight')
    plt.show()
    print('Saved:',filename)
    return field,output

field, optical = produce_main_figure()

## Simulate a rotating analyzer

In [ ]:
# Compare the optical appearance of the SAME stress field at 4 analyzer angles.
fig,axes=plt.subplots(2,2,figsize=(12.5,5.9),layout='constrained')
extent=(-WINDOW_LENGTH_MM/2,WINDOW_LENGTH_MM/2,-WINDOW_WIDTH_MM/2,WINDOW_WIDTH_MM/2)
for ax, angle in zip(axes.ravel(), (0,45,90,135)):
    linear=optical_images(field,angle)['rgb_linear']
    ax.imshow(linear**DISPLAY_GAMMA,extent=extent,origin='lower',vmin=0,vmax=1)
    ax.set_title(f'Analyzer {angle}°' + (' — parallel' if angle==0 else ' — crossed' if angle==90 else ''))
    ax.set_xlabel('x (mm)');ax.set_ylabel('y (mm)')
    add_hole_outline(ax)
fig.suptitle('Rotating-analyzer virtual camera images — idealized model',fontsize=13)
fig.savefig('POLARIS_X_Analyzer_Sweep.png',dpi=155,bbox_inches='tight')
plt.show()
print('Saved: POLARIS_X_Analyzer_Sweep.png')

## Run numerical and physics consistency checks

In [ ]:
# Scientifically motivated self-checks; this code intentionally stops if a check fails.
from numpy.testing import assert_allclose

# At r=a, theta=90°, the Kirsch hoop stress is 3 times remote uniaxial tension.
S = field['remote_mpa']
a = HOLE_RADIUS_MM
q = 1.0
sigma_tt_at_top = S/2*(1+q) - S/2*(1+3*q*q)*np.cos(np.pi)
assert_allclose(sigma_tt_at_top,3*S,rtol=1e-12)

# Zero load => zero stress and no birefringence; crossed analyzers block it.
zero_field=kirsch_stress_field(0)
assert np.max(np.abs(zero_field['delta_sigma'])) < 1e-12
assert np.max(optical_images(zero_field,90)['rgb_linear']) < 1e-12

# Hole (clear air) is dark under crossed polars and bright under parallel polars.
hole = ~field['outside']
assert np.max(optical_images(field,90)['rgb_linear'][hole]) < 1e-12
assert_allclose(optical_images(field,0)['rgb_linear'][hole],1.0,atol=1e-12)

# For crossed linear polarizers, Jones intensity must match the closed-form equation.
D = optical['retardances'][1]
predicted = (np.sin(2*field['phi'])**2)*(np.sin(D/2)**2)
assert_allclose(optical['rgb_linear'][:,:,1],predicted,atol=1e-12)

# Ideal normalized intensities have finite values bounded in [0,1].
for angle in (0,30,45,90,135,180):
    v=optical_images(field,angle)['rgb_linear']
    assert np.all(np.isfinite(v)) and v.min() >= 0 and v.max() <= 1

# Check stress field converges toward the far-field tensile state as r/a gets large.
far = kirsch_stress_field(FORCE_N,nx=551,ny=241)
end_idx = -1
assert abs(far['sigma_x'][far['sigma_x'].shape[0]//2,end_idx]-S)/S < .05

print('PASS: Kirsch hole boundary 3σ stress concentration')
print('PASS: zero load and crossed-analyzer darkness')
print('PASS: hole air-path light transmission')
print('PASS: Jones vs closed-form crossed-polarizer result')
print('PASS: finite, physically bounded intensity for 6 analyzer angles')
print('PASS: far-field stress consistency check')

## How to run and adapt

1. Open **https://colab.research.google.com** and choose **File → Upload notebook**; upload this `.ipynb` file.
2. Choose **Runtime → Run all** (shortcut: Ctrl+F9).
3. Look below the code cells for the figures and the `PASS` verification messages.
4. In the left sidebar select **Files** (folder icon). Download `POLARIS_X_Main_Results.png` and `POLARIS_X_Analyzer_Sweep.png`.
5. To test more force, change `FORCE_N = 1000.0` to `1500.0`, rerun all, and compare. To rotate the analyzer, change `ANALYZER_DEG = 90.0` to `45.0`.

### What to explain to your professor

> We developed an initial physics-based optical forward simulation. It starts from the analytical Kirsch stress field around a circular hole in an ideal infinite plate. We calculate local principal stress differences, convert them to optical phase retardation using an assumed stress-optic coefficient, and simulate intensity under different analyzer angles and illumination wavelengths. Next, we will replace the analytical stress field with a geometry-specific FEA model (ANSYS or FreeCAD) and experimentally calibrate the optics.

### Next stage — **not implemented in this notebook**

- Model a *finite* PMMA plate with real boundary conditions, holes, fixtures and load distribution using FEA.
- Export a grid of `x, y, σx, σy, τxy`; import that field into the notebook.
- Measure the actual material stress-optic coefficient and camera/illumination response.
- Implement inverse reconstruction and compare reconstructed principal stress differences to FEA, using held-out cases and explicit uncertainty.

**Limitations:** This notebook does not simulate deformation fields, a mechanically loaded finite plate, camera shot noise, quarter-wave plates, true stress reconstruction, defects unrelated to the circular hole, failure prediction, or a working physical digital twin. Static visual demonstration only, with an analytical stress model.
